## Qwen3.5 9B Distillation 
#### KNN Project 2026
TODO: 
- Sequence-level distillation / Top-k teacher distillation / Intermediate layer distillation

### Dependencies installation:

In [1]:
!pip install -q transformers accelerate datasets peft bitsandbytes trl
!pip install flash-attn --no-build-isolation

In [2]:
import torch
import torch.nn.functional as F

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

from datasets import load_dataset
from peft import LoraConfig, get_peft_model
from torch.utils.data import DataLoader

### Loading the teacher model:

In [3]:
import os
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
torch.cuda.empty_cache()

# Set environment variable to potentially mitigate OutOfMemoryError due to fragmentation
os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'

# Model Configurations
teacher_name = "Qwen/Qwen3.5-9B"

tokenizer = AutoTokenizer.from_pretrained(teacher_name)
tokenizer.pad_token = tokenizer.eos_token

# Quantization configuration
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

teacher = AutoModelForCausalLM.from_pretrained(
    teacher_name,
    quantization_config=bnb_config,
    device_map="auto",
    attn_implementation="flash_attention_2"
)

teacher.eval()

for p in teacher.parameters():
    p.requires_grad = False

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/427 [00:00<?, ?it/s]

### Loading the Dataset and splitting into teacher-finetuning and distillation subsets

In [4]:
finetune_dataset = load_dataset(
    "parquet",
    data_files="/workspace/czech_news_sft_100k.parquet",
    split="train[:50000]"
)

### Tokenization, Distillation Loss and ADAM:

In [5]:
def tokenize_and_mask(x, tokenizer, max_length=2048):
    full_article = x['content']
    brief = x['brief']

    # Prompt definition
    system_prompt = f"Jsi profesionální český novinář a editor. Tvým úkolem je vytvořit výstižné, přesné a neutrální shrnutí dodaného novinového článku. Shrnutí by mělo obsahovat nejdůležitější fakta a nesmí si vymýšlet žádné nové informace."
    user_prompt = f"Přečti si následující článek a napiš jeho stručné shrnutí (maximálně 2 až 3 věty).\n\nČLÁNEK:\n{full_article}:"

    system_text = f"<|im_start|>system\n{system_prompt}<|im_end|>\n"
    user_text = f"<|im_start|>user\n{user_prompt}<|im_end|>\n"
    assistant_text = f"<|im_start|>assistant\n{brief}<|im_end|>"

    # Tokenize the prompt components
    system_tokens = tokenizer(system_text, add_special_tokens=False)
    user_tokens = tokenizer(user_text, add_special_tokens=False)
    assistant_tokens = tokenizer(assistant_text, add_special_tokens=False)

    input_ids = (
        system_tokens["input_ids"]
        + user_tokens["input_ids"]
        + assistant_tokens["input_ids"]
        + [tokenizer.eos_token_id]
    )

    # Filter out longer entries
    if len(input_ids) > 1536:
        return None

    # Compute attention and loss mask
    # Loss: 0 for article and prompt components
    # 1 for the article brief 
    attention_mask = [1] * len(input_ids)

    loss_mask = (
        [0] * (len(system_tokens["input_ids"]) + len(user_tokens["input_ids"]))
        + [1] * (len(assistant_tokens["input_ids"]) + 1)
    )

    input_ids = input_ids[:max_length]
    attention_mask = attention_mask[:max_length]
    loss_mask = loss_mask[:max_length]

    # Label masking
    labels = input_ids.copy()
    labels[:len(system_tokens["input_ids"]) + len(user_tokens["input_ids"])] = [-100] * (
        len(system_tokens["input_ids"]) + len(user_tokens["input_ids"])
    )
    
    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels,
        "loss_mask": loss_mask,
    }
    

# Apply mapping
finetune_dataset_tokenized = finetune_dataset.map(lambda x: tokenize_and_mask(x, tokenizer))
finetune_dataset_tokenized = finetune_dataset_tokenized.filter(lambda x: x is not None)

finetune_dataset_tokenized.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"]
)

finetune_loader = DataLoader(finetune_dataset_tokenized, batch_size=1, shuffle=True)

Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/46913 [00:00<?, ? examples/s]

### Fine-tuning the Teacher Model using LoRA

In [6]:
import torch, trl, gc
from peft import LoraConfig, get_peft_model
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

# Configure LoRA for the teacher model
teacher_lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", 
                    "k_proj", 
                    "v_proj", 
                    "o_proj"
                   ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

teacher_peft = get_peft_model(teacher, teacher_lora_config)
# Print trainable parameters for confirmation
teacher_peft.print_trainable_parameters()


# Quick test to ensure model fits and forward pass works
gpu_stats = torch.cuda.get_device_properties(0)
reserved = torch.cuda.memory_reserved(0) / 1024**3
allocated = torch.cuda.memory_allocated(0) / 1024**3
free = (gpu_stats.total_memory / 1024**3) - reserved

print(f"GPU: {gpu_stats.name}")
print(f"Total Memory: {gpu_stats.total_memory / 1024**3:.2f} GB")
print(f"Allocated: {allocated:.2f} GB")
print(f"Free: {free:.2f} GB")

test_input = tokenizer("Testovací vstup", return_tensors="pt").to("cuda")

print("Testing Teacher...")
with torch.no_grad():
    t_out = teacher(**test_input)
print("Teacher OK.")

del t_out
torch.cuda.empty_cache()


data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
# data_collator = DataCollatorForLanguageModeling(finetune_dataset_tokenized, response_template="<|im_start|>assistant")

# Training arguments
training_args = TrainingArguments(
    output_dir="./teacher_finetuning_output", # Output directory for model checkpoints
    per_device_train_batch_size=1, # Batch size per GPU/CPU for training
    gradient_accumulation_steps=4, # Number of updates steps to accumulate before performing a backward/update pass
    warmup_steps=100, # Number of warmup steps for learning rate scheduler
    max_steps=1000, # Total number of training steps to perform
    learning_rate=2e-4, # Learning rate
    logging_dir="./teacher_finetuning_logs", # Directory for storing logs
    logging_steps=10, # Log every N updates steps
    save_steps=100, # Save checkpoint every N updates steps
    fp16=True, # Use mixed precision training
    optim="paged_adamw_8bit", # Optimizer
)

training_args = TrainingArguments(
    output_dir="./teacher_finetuning_output",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    max_steps=1000,
    learning_rate=2e-4,
    logging_steps=10,
    save_steps=100,
    bf16=True,
    fp16=False,
    gradient_checkpointing=True,
    optim="paged_adamw_8bit"
)

teacher_peft.gradient_checkpointing_enable()
teacher_peft.config.use_cache = False
# Create Trainer instance
teacher_trainer = Trainer(
    model=teacher_peft,
    args=training_args,
    train_dataset=finetune_dataset_tokenized, 
    data_collator=data_collator,
)

# Start training
teacher_trainer.train()
# Saves the LoRA adapters and the training configuration
teacher_trainer.save_model("./finetuned_3.5_teacher_model")
tokenizer.save_pretrained("./finetuned_3.5_teacher_model")
!zip -r finetuned_qwen3.5_teacher.zip /workspace/finetuned_3.5_teacher_model

trainable params: 1,966,080 || all params: 8,955,769,344 || trainable%: 0.0220
GPU: NVIDIA RTX A6000
Total Memory: 47.53 GB
Allocated: 7.13 GB
Free: 40.31 GB
Testing Teacher...


`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Teacher OK.


Step,Training Loss


KeyboardInterrupt: 

In [ ]:
!zip -r finetuned_qwen3.5_teacher.zip /workspace/finetuned_3.5_teacher_model

### Testing Finetuned Teacher and Base Student Model Performance

In [ ]:
import random

# Select a random entry from the dataset for testing
random_idx = random.randint(0, len(finetune_dataset_tokenized) - 1)

device = "cuda" if torch.cuda.is_available() else "cpu"

example = finetune_dataset_tokenized[random_idx]
# Extract the full prompt used for training and the reference brief
content = finetune_dataset[random_idx]['content']
reference_brief = finetune_dataset[random_idx]['brief']

system_prompt = "Jsi profesionální český novinář a editor. Tvým úkolem je vytvořit výstižné, přesné a neutrální shrnutí dodaného novinového článku. Shrnutí by mělo obsahovat nejdůležitější fakta a nesmí si vymýšlet žádné nové informace."
user_prompt = f"Přečti si následující článek a napiš jeho stručné shrnutí (maximálně 2 až 3 věty).\n\nČLÁNEK:\n{content}"
prompt = (
    f"<|im_start|>system\n{system_prompt}<|im_end|>\n"
    f"<|im_start|>user\n{user_prompt}<|im_end|>\n"
    f"<|im_start|>assistant\n"
)

print(f"--- Original Article ---\n{content}\n")
print(f"--- Reference Summary ---\n{reference_brief}\n")

# Tokenize the prompt for generation
inputs = tokenizer(
    prompt,
    return_tensors="pt",
    add_special_tokens=False
).to(device)

# Clear out cache between running teacher model
torch.cuda.empty_cache()
teacher_peft.eval()
with torch.no_grad():
    outputs = teacher_peft.generate(
        **inputs,
        max_new_tokens=200,
        do_sample=True,          # Sampling instead of greedy decoding
        temperature=0.6,         # Creativity rate
        top_p=0.8,
        repetition_penalty=1.1,  # Penalty to prevent repetition collapse
        pad_token_id=tokenizer.eos_token_id
    )

# Decode Generated tokens
generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]
generated_summary = tokenizer.decode(
    generated_tokens,
    skip_special_tokens=True
)
print(f"--- Generated Summary (Teacher Model) ---\n{generated_summary}\n")